# npy2pointcloud Demo

## What is npy2pointcloud?

The [Rohbau3D](https://huggingface.co/datasets/Finnish-NLP/Rohbau3D) dataset stores point clouds as a set of `.npy` files per scene:

| File | Shape | Description |
|------|-------|-------------|
| `coord.npy` | (N, 3) float | XYZ coordinates (required) |
| `color.npy` | (N, 3) uint8 | RGB values |
| `intensity.npy` | (N, 1) float | Laser reflectance |
| `normal.npy` | (N, 3) float | Surface normals |

While this format is convenient for NumPy workflows, standard tools like **Open3D** and **CloudCompare** expect PLY, PCD, or LAS files.

`npy2pointcloud` bridges this gap by converting Rohbau3D `.npy` directories into these standard formats, preserving colors, normals, and intensity.

## 1. Create Sample Data

We generate a synthetic building interior scene (~5000 points) with floor, walls, and ceiling.

In [ ]:
import numpy as np
from pathlib import Path
import tempfile
import shutil

# Create a temporary workspace
work_dir = Path(tempfile.mkdtemp(prefix="npy2pc_demo_"))
scene_dir = work_dir / "scene_001"
scene_dir.mkdir(parents=True, exist_ok=True)

np.random.seed(42)

# Room dimensions: 6m x 4m x 3m
room_w, room_d, room_h = 6.0, 4.0, 3.0

def make_plane(n, x_range, y_range, z_range, jitter=0.01):
    """Generate points on a plane with slight noise."""
    pts = np.zeros((n, 3))
    pts[:, 0] = np.random.uniform(*x_range, n)
    pts[:, 1] = np.random.uniform(*y_range, n)
    pts[:, 2] = np.random.uniform(*z_range, n)
    pts += np.random.normal(0, jitter, pts.shape)
    return pts

# Floor (z ~ 0)
floor = make_plane(1500, (0, room_w), (0, room_d), (0, 0))
floor_color = np.tile([180, 170, 150], (1500, 1))  # beige

# Ceiling (z ~ 3)
ceiling = make_plane(1000, (0, room_w), (0, room_d), (room_h, room_h))
ceiling_color = np.tile([240, 240, 240], (1000, 1))  # white

# Wall left (x ~ 0)
wall_l = make_plane(600, (0, 0), (0, room_d), (0, room_h))
wall_l_color = np.tile([200, 200, 210], (600, 1))  # light blue-gray

# Wall right (x ~ 6)
wall_r = make_plane(600, (room_w, room_w), (0, room_d), (0, room_h))
wall_r_color = np.tile([200, 200, 210], (600, 1))

# Wall back (y ~ 0)
wall_b = make_plane(700, (0, room_w), (0, 0), (0, room_h))
wall_b_color = np.tile([210, 200, 190], (700, 1))  # warm gray

# Wall front (y ~ 4)
wall_f = make_plane(600, (0, room_w), (room_d, room_d), (0, room_h))
wall_f_color = np.tile([210, 200, 190], (600, 1))

# Combine all points
coords = np.vstack([floor, ceiling, wall_l, wall_r, wall_b, wall_f])
colors = np.vstack([floor_color, ceiling_color, wall_l_color, wall_r_color,
                     wall_b_color, wall_f_color]).astype(np.uint8)

n_points = coords.shape[0]

# Generate intensity (simulated laser reflectance)
intensity = np.random.uniform(0.2, 0.9, (n_points, 1)).astype(np.float32)

# Generate normals (approximate per-surface normals with noise)
normals = np.zeros((n_points, 3))
idx = 0
for n_pts, normal_vec in [(1500, [0, 0, 1]),    # floor -> up
                           (1000, [0, 0, -1]),   # ceiling -> down
                           (600, [1, 0, 0]),     # wall left -> +x
                           (600, [-1, 0, 0]),    # wall right -> -x
                           (700, [0, 1, 0]),     # wall back -> +y
                           (600, [0, -1, 0])]:   # wall front -> -y
    normals[idx:idx+n_pts] = normal_vec
    idx += n_pts
normals += np.random.normal(0, 0.02, normals.shape)
# Re-normalize
normals /= np.linalg.norm(normals, axis=1, keepdims=True)

# Save as Rohbau3D-style .npy files
np.save(scene_dir / "coord.npy", coords)
np.save(scene_dir / "color.npy", colors)
np.save(scene_dir / "intensity.npy", intensity)
np.save(scene_dir / "normal.npy", normals)

print(f"Scene directory: {scene_dir}")
print(f"Total points:    {n_points:,}")
print(f"Files created:   {[f.name for f in sorted(scene_dir.glob('*.npy'))]}")

## 2. Inspect Data

Use the `npy2pointcloud info` CLI command to view point cloud statistics.

In [ ]:
!npy2pointcloud info -i {scene_dir}

We can also use the Python API directly:

In [ ]:
from npy2pointcloud.loader import load_scene

data = load_scene(scene_dir)
print(data.summary())
print(f"\nHas colors:    {data.has_colors}")
print(f"Has intensity: {data.has_intensity}")
print(f"Has normals:   {data.has_normals}")

## 3. Convert to PLY

In [ ]:
from npy2pointcloud.converter import convert

output_dir = work_dir / "output"
output_dir.mkdir(exist_ok=True)

ply_path = convert(data, output_dir / "scene_001", "ply")
print(f"PLY file: {ply_path}")
print(f"File size: {ply_path.stat().st_size / 1024:.1f} KB")

In [ ]:
# Verify by reading back with Open3D
import open3d as o3d

pcd_loaded = o3d.io.read_point_cloud(str(ply_path))
print(f"Points read back: {len(pcd_loaded.points):,}")
print(f"Has colors:  {pcd_loaded.has_colors()}")
print(f"Has normals: {pcd_loaded.has_normals()}")

## 4. Convert to PCD

In [ ]:
pcd_path = convert(data, output_dir / "scene_001", "pcd")
print(f"PCD file: {pcd_path}")
print(f"File size: {pcd_path.stat().st_size / 1024:.1f} KB")

# Verify
pcd_loaded = o3d.io.read_point_cloud(str(pcd_path))
print(f"Points read back: {len(pcd_loaded.points):,}")

## 5. Convert to LAS

In [ ]:
las_path = convert(data, output_dir / "scene_001", "las")
print(f"LAS file: {las_path}")
print(f"File size: {las_path.stat().st_size / 1024:.1f} KB")

# Verify with laspy
import laspy
las = laspy.read(str(las_path))
print(f"\nLAS version: {las.header.version}")
print(f"Point format: {las.header.point_format.id}")
print(f"Points read back: {las.header.point_count:,}")
print(f"Has RGB: {hasattr(las, 'red')}")
print(f"Has intensity: {hasattr(las, 'intensity')}")
print(f"Extra dims: {[d.name for d in las.point_format.extra_dims]}")

## 6. Visualize

Since Jupyter notebooks cannot render Open3D interactive windows, we use matplotlib for 3D scatter plots.
We show the point cloud from two views: top-down (XY) and side (XZ).

In [ ]:
import matplotlib.pyplot as plt

# Prepare colors for matplotlib (normalize to [0, 1])
plot_colors = data.colors.astype(np.float64) / 255.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# XY view (top-down)
ax = axes[0]
ax.scatter(data.coords[:, 0], data.coords[:, 1],
           c=plot_colors, s=0.5, alpha=0.6)
ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_title("Top-down view (XY)")
ax.set_aspect("equal")

# XZ view (side)
ax = axes[1]
ax.scatter(data.coords[:, 0], data.coords[:, 2],
           c=plot_colors, s=0.5, alpha=0.6)
ax.set_xlabel("X (m)")
ax.set_ylabel("Z (m)")
ax.set_title("Side view (XZ)")
ax.set_aspect("equal")

plt.suptitle(f"Building Interior Scene ({data.num_points:,} points)", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Batch Conversion

Create multiple scenes and convert them all at once.

In [ ]:
# Create two additional scenes
for i, (w, d, h) in enumerate([(5.0, 3.0, 2.5), (8.0, 6.0, 3.5)], start=2):
    sdir = work_dir / "dataset" / f"scene_{i:03d}"
    sdir.mkdir(parents=True, exist_ok=True)
    n = 1000
    # Simple box: floor + 4 walls
    pts = np.vstack([
        make_plane(n, (0, w), (0, d), (0, 0)),      # floor
        make_plane(n // 4, (0, 0), (0, d), (0, h)),  # wall
        make_plane(n // 4, (w, w), (0, d), (0, h)),  # wall
        make_plane(n // 4, (0, w), (0, 0), (0, h)),  # wall
        make_plane(n // 4, (0, w), (d, d), (0, h)),  # wall
    ])
    col = np.random.randint(100, 240, (pts.shape[0], 3), dtype=np.uint8)
    np.save(sdir / "coord.npy", pts)
    np.save(sdir / "color.npy", col)
    print(f"Created {sdir.name}: {pts.shape[0]} points")

# Also copy scene_001 into the dataset directory
dst = work_dir / "dataset" / "scene_001"
shutil.copytree(scene_dir, dst)

In [ ]:
# Batch convert using the CLI
batch_out = work_dir / "batch_output"
!npy2pointcloud batch -i {work_dir / 'dataset'} -o {batch_out} -f ply -v

In [ ]:
# List the output files
for p in sorted(batch_out.rglob("*.ply")):
    rel = p.relative_to(batch_out)
    print(f"  {rel}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Clean up temporary files
shutil.rmtree(work_dir)
print(f"Cleaned up {work_dir}")

## Summary

### Supported Output Formats

| Format | Extension | Library | Colors | Normals | Intensity | Notes |
|--------|-----------|---------|--------|---------|-----------|-------|
| PLY    | `.ply`    | Open3D  | Yes    | Yes     | --        | Binary format, widely supported |
| PCD    | `.pcd`    | Open3D  | Yes    | Yes     | --        | Point Cloud Library native format |
| LAS    | `.las`    | laspy   | Yes (16-bit) | Yes (extra dims) | Yes | LAS 1.4, point format 7 |

### Links

- [Rohbau3D dataset on HuggingFace](https://huggingface.co/datasets/Finnish-NLP/Rohbau3D)
- [npy2pointcloud on GitHub](https://github.com/rsasaki0109/npy2pointcloud)
- [Open3D documentation](http://www.open3d.org/docs/)
- [CloudCompare](https://www.cloudcompare.org/)